In [14]:
import json
data = []
for i in range(2):
    with open(f"darija_worker{i}.jsonl","r",encoding='utf-8') as file:
        for line in file:
            # Parse each line as a JSON object
            json_object = json.loads(line.strip())
            data.append(json_object)
len(data)

121660

In [2]:
import pandas as pd
df = pd.DataFrame(data)
df.head()

,id,en,fr,darija
0,0,The Wanderer,Le grand Meaulnes,اللي كيدور
1,1,Alain-Fournier,Alain-Fournier,ألان فورنييه
2,2,First Part,PREMIÈRE PARTIE,الجزء الأول
3,3,I,CHAPITRE PREMIER,أنا
4,4,THE BOARDER,LE PENSIONNAIRE,اللي كاري بيت


In [3]:
df.duplicated(subset="id").sum()

np.int64(5)

In [4]:
from datasets import load_dataset
ds = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

In [5]:
ids_list = df["id"].to_list()
filtred_ds = ds.filter(
    lambda x: int(x["id"]) in ids_list
)
filtred_ds

Filter:   0%|          | 0/127085 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'translation'],
    num_rows: 119555
})

In [6]:
filtred_ds_list = []
for row in filtred_ds:
    row["translation"]["id"] = int(row["id"])
    filtred_ds_list.append(
        row["translation"]
    )

In [7]:
ds_df = pd.DataFrame(filtred_ds_list)
ds_df.head()

,en,fr,id
0,The Wanderer,Le grand Meaulnes,0
1,Alain-Fournier,Alain-Fournier,1
2,First Part,PREMIÈRE PARTIE,2
3,I,CHAPITRE PREMIER,3
4,THE BOARDER,LE PENSIONNAIRE,4


In [8]:
result = pd.merge(ds_df, df[["id","darija"]], on='id')
result.head()

,en,fr,id,darija
0,The Wanderer,Le grand Meaulnes,0,اللي كيدور
1,Alain-Fournier,Alain-Fournier,1,ألان فورنييه
2,First Part,PREMIÈRE PARTIE,2,الجزء الأول
3,I,CHAPITRE PREMIER,3,أنا
4,THE BOARDER,LE PENSIONNAIRE,4,اللي كاري بيت


In [9]:
result.sample(13)

,en,fr,id,darija
90425,"The reporter and his companions, after having ...","Le reporter et ses compagnons, après avoir abs...",96257,الصحفي وصحابو، من بعد ما كلو بزاف ديال الصدفات...
50613,She could not detach her eyes from the carpet ...,Elle ne pouvait détacher sa vue de ce tapis où...,50608,ما قدراتش تحيد عينيها من الزربية فين كان كيتمش...
109802,It was a known fact that when a miner wished t...,C'était un fait connu: quand un mineur voulait...,115634,كانت معروفة بلي أي مناجم بغا يطول الكريدي ديال...
59395,We said it was the funniest thing we had ever ...,"On n’avait jamais rien entendu de plus drôle, ...",65227,قلنا بلي هادي أضحك حاجة سمعناها فحياتنا كاملة.
30701,"If he isn't absolutely stale, Tregellis, he is...","S'il n'est pas absolument vanné, Tregellis, il...",30696,"""إلا ما كانش عيا بزاف يا تريجيليس، راه هو أحسن..."
57871,It was the hour when the earliest windows of t...,C’était l’heure où les fenêtres les plus matin...,63703,كانت ديك الساعة فاش كيبداو يتحلو الشراجم اللول...
36507,"""Yes, sir.""","--Oui, monsieur.",36502,"""إيه أ سيدي."""
52407,He muddled up the stage-boxes with the gallery...,"Il confondit l’avant-scène avec les galeries, ...",52402,تخلطو ليه البلايص ديال المسرح، بين البالكونات ...
7103,"When first Mr. Bennet had married, economy was...","Quand Mr. Bennet s’était marié, il n’avait pas...",7098,ملي تزوج السيد بينيت أول مرة، كان كيسحاب ليه ب...
117662,"She had already allowed three cages to pass, a...","Elle avait déja laissé passer trois cages, ell...",123494,خلات تلاتة ديال القفاص يدوزو، وقالت، بحال يلا ...


In [10]:
from datasets import Dataset
new_dataset = Dataset.from_pandas(result)
new_dataset

Dataset({
    features: ['en', 'fr', 'id', 'darija'],
    num_rows: 119560
})

In [12]:
new_dataset = new_dataset.select_columns(["id","en","fr","darija"])
new_dataset

Dataset({
    features: ['id', 'en', 'fr', 'darija'],
    num_rows: 119560
})